# Entropic regularization on a transport polytope

This notebook generates `fig:sinkhorn-entropy-lp-geometry`.  On a triangular face of a transport polytope, the unregularized objective is linear and selects a vertex.  Entropic regularization adds a barrier: for every positive `epsilon`, the minimizer stays in the relative interior and follows a smooth path toward the low-cost vertex as `epsilon` decreases.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.patches import Polygon

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    ORANGE,
    RED,
    VIOLET,
    box_axes,
    canonical_matching_clouds,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
np.random.seed(0)

## A triangular face

The three barycentric coordinates represent three extreme couplings.  For a reference measure `q`, the entropic minimizer has the closed form
$$
    p_\varepsilon \propto q \odot \exp(-c/\varepsilon).
$$
The notebook exports three individual temperature panels and a separate path panel.

In [ ]:
fig_name = "sinkhorn-entropy-lp-geometry"
out = figure_dir(fig_name)

vertices = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])
cost = np.array([0.0, 0.58, 1.15])
reference = np.array([1.0, 1.0, 1.0]) / 3


def bary_to_xy(p):
    return p @ vertices


def softmin_point(epsilon):
    w = reference * np.exp(-cost / epsilon)
    return w / w.sum()

resolution = 90
bary = []
values = []
for i in range(resolution + 1):
    for j in range(resolution + 1 - i):
        p = np.array([i, j, resolution - i - j], dtype=float) / resolution
        bary.append(p)
        values.append(float(np.dot(cost, p)))
bary = np.asarray(bary)
points = bary_to_xy(bary)
values = np.asarray(values)

eps_panels = [(1.10, "eps-large.pdf"), (0.35, "eps-medium.pdf"), (0.095, "eps-small.pdf")]
path_eps = np.geomspace(1.35, 0.035, 44)
path = np.vstack([softmin_point(eps) for eps in path_eps])
path_xy = bary_to_xy(path)

## Exported panels

No title or numerical label is embedded in the PDF.  The LaTeX caption names the three temperatures and explains the violet path.

In [ ]:
def draw_triangle_base(ax):
    poly = Polygon(vertices, closed=True, facecolor="#fff7e6", edgecolor=GRAY, linewidth=0.85, zorder=1)
    ax.add_patch(poly)
    ax.tricontour(points[:, 0], points[:, 1], values, levels=np.linspace(0.16, 1.02, 6), colors=GRAY, linewidths=0.50, alpha=0.40, zorder=2)
    ax.scatter(vertices[:, 0], vertices[:, 1], s=18, marker="o", color=[VIOLET, LIGHT_GRAY, LIGHT_GRAY], edgecolor="none", zorder=3)
    ax.set_xlim(-0.08, 1.08)
    ax.set_ylim(-0.07, np.sqrt(3) / 2 + 0.09)
    ax.set_aspect("equal")
    remove_axes(ax)


def draw_temperature_panel(epsilon, filename):
    p = softmin_point(epsilon)
    xy = bary_to_xy(p)
    fig, ax = plt.subplots(figsize=(2.02, 1.86))
    draw_triangle_base(ax)
    ax.scatter([xy[0]], [xy[1]], s=30, marker="o", color=VIOLET, edgecolor="none", zorder=5)
    for vertex in vertices:
        ax.plot([xy[0], vertex[0]], [xy[1], vertex[1]], color=VIOLET, lw=0.45, alpha=0.18, zorder=4)
    save_pdf(fig, out / filename, pad_inches=0.045)
    plt.close(fig)


for epsilon, filename in eps_panels:
    draw_temperature_panel(epsilon, filename)

fig, ax = plt.subplots(figsize=(2.20, 1.86))
draw_triangle_base(ax)
colors = [interp_color(t, ORANGE, VIOLET) for t in np.linspace(0, 1, len(path_xy))]
ax.plot(path_xy[:, 0], path_xy[:, 1], color=VIOLET, linewidth=1.12, alpha=0.82, zorder=4)
ax.scatter(path_xy[:, 0], path_xy[:, 1], s=11, marker="o", color=colors, edgecolor="none", zorder=5)
save_pdf(fig, out / "path.pdf", pad_inches=0.045)
plt.close(fig)